# Bubble Bi

### Reading the stock market like a language

## The problem

Every day, a stock leaves a trace: open, high, low, close, volume. Markets repeat certain moods — calm drift, panic, sharp reversal — but nobody labels them, and there is no dictionary of them.

**The idea:** let the machine build that dictionary itself.

We compress each stock-day into a single **token** — one word out of 512 that the machine invents on its own. A stock's history then becomes a sentence:

```
AAPL: ... 147  147  391  208  208  63  →  what comes next?
```

From there it is the same trick as ChatGPT: read the sentence, predict the next word.

**Three stages:**

| | | |
|---|---|---|
| 1. **Tokenizer** | turns each stock-day into one token | ✅ built |
| 2. **Predictor** | a GPT that predicts the next token | ✅ built |
| 3. **Trader** | acts on it: long / short / flat | ⬜ not built |

> ⚠️ **The one rule:** nothing may ever look into the future. Break it and the model looks brilliant while losing real money. A test enforces it.

Run the cells below in order.

---
## 1. Settings

Every knob is here — nothing hidden in a config file.

**The tokenizer looks at each day through two windows**, then merges what it sees into a single token:

```
  TS  — what THIS stock has been doing   (one stock, over time)   ┐
                                                                  ├──→  ONE token
  CS  — what the WHOLE MARKET was doing  (all stocks, on a day)   ┘
```

Each window has its own settings.

In [ ]:
SETTINGS = dict(

    # ── Which companies, and how far back ────────────────────────────────
    tickers = ["AAPL", "MSFT", "AMZN", "GOOGL", "META", "NVDA", "JPM", "V", "JNJ", "WMT",
               "PG", "HD", "BAC", "XOM", "CVX", "KO", "PEP", "DIS", "CSCO", "INTC",
               "VZ", "T", "MRK", "PFE", "ABT", "NKE", "MCD", "CAT", "BA", "IBM"],
    start   = "2010-01-01",

    # ── ENTRY 1 — TS: what THIS stock has been doing ─────────────────────
    ts = dict(
        days          = 4,     # days of the stock's own history it reads
        vocabulary    = 512,   # size of its private dictionary
        encoder_depth = 3,     # how deep it looks
        decoder_depth = 2,     # (only used while training, to check it understood)
    ),

    # ── ENTRY 2 — CS: what the WHOLE MARKET was doing ────────────────────
    cs = dict(
        days          = 5,     # days of market-wide history it reads
        vocabulary    = 512,
        encoder_depth = 3,
        decoder_depth = 2,
    ),

    # ── Where the two entries merge into the ONE token we keep ───────────
    fusion = dict(
        vocabulary = 512,      # THIS is the dictionary the predictor actually reads
        depth      = 2,
    ),

    # ── The predictor: the GPT that reads the sentences ──────────────────
    predictor = dict(
        sentence_length = 64,  # how many past tokens it reads before guessing
        depth           = 4,
    ),

    # ── Shared ───────────────────────────────────────────────────────────
    model_size = 128,   # brain width. Shared on purpose: TS and CS meet in a
                        # cross-attention layer, which needs both the same width.

    # ── Training ─────────────────────────────────────────────────────────
    steps         = 2000,   # how long to train (raise this on a GPU)
    batch_size    = 256,
    learning_rate = 1e-4,
    seed          = 42,     # same seed = same result, every time
)

---
## 2. Get the code

Downloads the project and installs what it needs. Safe to re-run — it skips anything already there.

In [ ]:
import importlib, os, subprocess, sys

REPO   = "https://github.com/hockper/Quant-AI-2026.git"
BRANCH = "notebook-first"

if not os.path.exists("bubble_bi"):                  # not here yet -> fetch it
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "bubble-bi"], check=True)
    os.chdir("bubble-bi")

sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

# Python caches imported code. Without this, re-running the notebook after the code
# changes would keep using the OLD copy still sitting in memory.
for loaded in [m for m in sys.modules if m == "bubble_bi" or m.startswith("bubble_bi.")]:
    del sys.modules[loaded]
importlib.invalidate_caches()

import bubble_bi as bb

settings = bb.check(SETTINGS)     # fills in defaults, catches typos
print(bb.summary(settings))

### ✅ Check

Every section of this notebook ends like this: it *proves* the step worked, then tells you what you now have. If something is wrong it stops here rather than letting you carry on.

In [ ]:
bb.verify.setup(settings)

---
## 3. Get the prices

Download every company's daily history and put it in one table.

Each row is **one company on one day**, holding the six numbers a trading day leaves behind — plus `target`, tomorrow's return.

> `target` is *the answer*. It is the only column allowed to look forward, and it is never given to a model as an input.

Downloads are cached, so this is slow once and instant afterwards.

In [ ]:
prices = bb.data.download(settings)

prices.head()

In [ ]:
bb.verify.prices(prices, settings)

---
## 4. Describe each day properly

Raw prices are a thin description of a trading day. Before the machine can find *moods*, we re-describe every day from five angles:

| | |
|---|---|
| **price** | where it is going — returns, moving averages, RSI, MACD |
| **volatility** | how violently it moves — four estimators, each reading a different part of the candle |
| **memory** | whether the past still echoes — Hurst, fractional differencing, entropy |
| **microstructure** | what it *costs* to trade — hidden spread, illiquidity |
| **flow** | who is pushing, and how hard — volume dynamics |

That's **22 numbers per company per day**. This table is what the tokenizer will learn to squeeze into single tokens.

In [ ]:
data = bb.data.add_features(prices, settings)

data.tail()

### ✅ Check

The important one. **We do not trust that our features are backward-looking — we prove it.**

The check below takes the data, **deletes the future**, recomputes every feature from scratch, and verifies that not a single past value changed. If any feature were secretly peeking at tomorrow, deleting tomorrow would change it, and this catches it.

This is the check that separates a real trading model from one that fools its author.

In [ ]:
bb.verify.features(data, prices, settings)

---
## 5. Build the two entries

Time for the machine that invents the dictionary.

**TS and CS look like two different problems. They aren't.** They are the same problem at two sizes — so they are the *same class*, configured differently:

```
TS     1 company  × 4 days × 22 features   →  one word     "what THIS stock is doing"
CS    30 companies × 5 days × 22 features  →  one word     "what the MARKET is doing"
```

TS is simply CS watching a single company. Each one works the same way:

| | |
|---|---|
| **encoder** | reads the whole grid and boils it down to a single vector |
| **codebook** | snaps that vector to the **nearest word** in its dictionary of 512 |
| **decoder** | tries to rebuild the *entire grid* from that one word |

Training makes the rebuild as accurate as possible. **That is the whole trick.** Nobody tells the machine what the words should mean — it is simply forced to describe a grid of market data using one word out of 512, and the only way to do that well is for the words to come to mean something.

In [ ]:
n_features  = len(bb.data.names())        # 22
n_companies = len(settings["tickers"])    # 30

# The SAME class, twice. Only the numbers differ.
ts = bb.models.VQVAE(companies=1,           features=n_features,
                     width=settings["model_size"], **settings["ts"])

cs = bb.models.VQVAE(companies=n_companies, features=n_features,
                     width=settings["model_size"], **settings["cs"])

print("TS :", ts.describe())
print("CS :", cs.describe())

### ✅ Check

Push a fake grid through both machines and watch what comes out. In particular: **gradients must survive the snap.**

Snapping a vector to its nearest word is a step function — it has no slope, so ordinarily no learning signal could pass back through it. A trick called *straight-through* smuggles the gradient past. If that were broken, training would appear to run, the loss would sit still, and nothing would tell you why.

In [ ]:
bb.verify.models(ts, cs, settings)

---
## 6. Cut the table into grids

The models eat grids, not tables. Cut them out, normalise them, and divide history into three periods:

| | |
|---|---|
| **learn** | the model trains on these days |
| **tune** | used to watch progress and stop at the right moment |
| **test** | untouched until the very end. The only honest score. |

**The split is strictly by date.** Learn on the past, tested on the future — the way it would actually have to work.

### Every company is normalised against *itself*

A \$500 stock and a \$20 stock are not comparable. Nor is a heavily-traded giant and a thin one. Left alone, **half of some features is nothing but *which company it is***:

```
close_frac   68% ← just the price level
amihud       41% ← just how liquid it is
obv_frac     39% ← just how many shares it trades
```

The codebook only has 512 words. If we hand it that, it will spend them memorising *"this is NVDA"* instead of *"this is a panic"*.

So each company is normalised **against its own history, down the time axis** — never against the other companies. Afterwards every feature says the same thing for everyone:

> *"How unusual is today — **for this company**?"*

⚠️ **What this deliberately throws away:** that one company is *genuinely* more volatile, or less liquid, than another. A sleepy utility at its own average volatility and a wild biotech at *its* own average volatility now look identical. That is the price of a word meaning the same thing whoever it describes.

> ### ⚠️ Three ways a trading model quietly cheats
>
> **1. Shuffling time.** Split days at random and the model trains on Thursday, then gets tested on Wednesday. We split by *date*.
>
> **2. Scaling with the future.** Compute that normalisation from *all* of history and the average of days you haven't seen yet leaks backwards into training. We measure it on the **learn period only**.
>
> **3. A target that reaches over the border.** The last learn day's target is *tomorrow's* return — and tomorrow is the first tune day. We drop that day. One row, and it's the difference between an honest boundary and a leaky one.
>
> The check below **proves** all of it, rather than promising it.

In [ ]:
batches = bb.data.make_tensors(data, settings)

batches.sizes()

In [ ]:
bb.verify.tensors(batches, ts, cs, settings)

---
## 7. Train TS — watch it invent a vocabulary

Give the model a grid, make it squeeze the grid into **one word**, then make it rebuild the grid from that word alone. Nobody says what the words should mean. The only way to rebuild well is for them to come to mean something.

### The number to watch is *not* the loss

It is the **perplexity** — how many words are actually in use.

A VQ-VAE has a nasty habit: a few words win everything early on, the rest are never chosen again, and the model settles for describing the whole market with a handful of words. **The loss can look respectable while this is happening.** Perplexity is what exposes it:

| perplexity | meaning |
|---|---|
| **≈ 1** | catastrophe — one word for everything, nothing learned |
| **≈ 50** | the vocabulary collapsed to a fraction of itself |
| **300+** | healthy — genuinely using its dictionary |

Watch the first line of the run below: perplexity **starts at 1.0**. The model really does collapse onto a single word. Two things pull it out:

- **Reviving dead words** — any word nobody is choosing gets dropped onto a real encoder output, so it has somewhere realistic to compete from.
- **A diversity penalty** — the model is punished for crowding onto a few words. (This is the STORM paper's own anti-collapse term; adding it nearly doubled the codebook usage in our tests.)

> ⏱️ **This is a short run**, so it finishes on a laptop CPU. Raise `steps` and run it on a GPU for the real thing.

**`rebuild`** is scored against a deliberately stupid baseline: *guessing the average for everything*. Because features were normalised to spread 1, that baseline scores ≈ 1.0. So `rebuild = 0.6` means the single token captured about **40%** of what was going on.

In [ ]:
QUICK = 400        # a laptop-sized run. Set to None to use settings["steps"] on a GPU.

ts_history = bb.train(ts, batches.ts, settings, steps=QUICK)

In [ ]:
bb.verify.trained(ts, ts_history, batches.ts, settings, name="TS")

---
## 8. Look at what one word remembered

Enough numbers. **Look at it.**

Take four days the model has never seen, squeeze them into a single word out of 512, and rebuild the candles from that word alone.

> **How can it draw a candle?** The model never sees a candle — raw prices trend and differ wildly between companies, and feeding them in would put drift and company-identity straight back. What it sees is the candle's **shape**: `gap`, `body`, `upper_wick`, `lower_wick` — all ratios, all scale-free. Given the previous close, those four numbers rebuild open/high/low/close **exactly**. So what you're looking at is precisely what the word remembered, not an artist's impression.

In [ ]:
bb.plots.remembered(ts, batches, prices, settings);

### What it kept, and what it threw away

The reconstruction above is *smoother* than reality. That is not a failure — it is the whole point of the bottleneck, and the chart below shows exactly what happened.

One word cannot carry everything. Forced to choose, the model keeps the **slow context** — the trend, the volatility regime, the momentum — and discards the **day-to-day wiggle**.

Which is remarkable, because we deliberately *gave* it the candle. **It looked at the candle and decided the candle was noise.**

In [ ]:
bb.plots.kept_and_lost(ts, batches, settings);

In [ ]:
bb.plots.progress(ts_history, "TS");

---
## 9. Train CS — the whole market in one word

Same machine, far harder job. TS describes **one company** over 4 days. CS has to describe **all 30 companies at once** — the mood of the entire market — and squeeze *that* into a single word.

### Expect a much weaker number, and know why

```
TS:    1 company  × 4 days × 26 features =   104 numbers  →  1 word
CS:   30 companies × 5 days × 26 features = 3,900 numbers  →  1 word
```

**CS is compressing 38× harder, from 30× less data** (one grid per *day*, not per company-day).

It will not rebuild the market well. It cannot: one word out of 512 has nowhere near enough room to say what thirty companies each did. Most of what a company does is its *own* business, and a single shared word can only carry what they have **in common** — the market-wide mood.

**That is fine, because rebuilding the market is not CS's job.** Its job is to learn a useful *market context*, which the fusion will hand to each company's token in the next section. The decoder and the codebook here are **scaffolding**: they exist to give the encoder something to learn from, and then we throw them away and keep the encoder.

So watch the **perplexity**, not the rebuild score. If the dictionary collapses to a handful of words, the encoder was trained through a broken bottleneck and learned little. That is the failure that matters.

> ⚠️ A short CPU run will look bad on both counts. Do not tune anything on these numbers — do it on Colab.

In [ ]:
cs_history = bb.train(cs, batches.cs, settings, steps=QUICK)

In [ ]:
bb.verify.trained(cs, cs_history, batches.cs, settings, name="CS")

bb.plots.progress(cs_history, "CS");